# CDLE: Diagnóstico e Profiling de Performance em Big Data

Este notebook é dedicado exclusivamente ao **profiling de CPU** das operações mais exigentes do pipeline (**Value Counts** e **GroupBy**) nas 5 frameworks: **Pandas, Dask, PySpark, Modin e Joblib**.

Utiliza-se a magia do Jupyter `%%prun -s tottime -l 15` no topo de cada célula de computação para listar e ordenar as 15 funções internas que mais consomem tempo de CPU, permitindo analisar bottlenecks de forma isolada e económica no Dataproc.

In [ ]:
import sys
!{sys.executable} -m pip install -q "modin[ray]" matplotlib


In [ ]:
# --- PATCH GLOBAL DE VISUALIZAÇÃO DE TABELAS ---
import pandas as pd
if not hasattr(pd.Index, '_format_flat'):
    pd.Index._format_flat = lambda self, *args, **kwargs: [str(x) for x in self]

try:
    import pyspark.pandas as ps
    if not hasattr(ps.Index, '_format_flat'):
        ps.Index._format_flat = lambda self, *args, **kwargs: [str(x) for x in self]
except Exception:
    pass
# -----------------------------------------------

import time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
print(" 1 - Escala Pequena (~700k linhas)")
print(" 2 - Escala Média (~14M linhas)")
print(" 3 - Escala Grande (~42M linhas)")

while True:
    try:
        op = input("Introduza a opcao desejada (1, 2 ou 3): ").strip()
        ESCALA_DATASET = int(op)
        if ESCALA_DATASET in [1, 2, 3]:
            break
        else:
            print("Opcao invalida. Por favor, introduza 1, 2 ou 3.")
    except ValueError:
        print("Entrada invalida. Por favor, introduza um numero.")

base_dir = "gs://dataproc-staging-europe-southwest1-348488791616-f80l4tzf/notebooks/jupyter/"
caminho = {
    1: base_dir + "yellow_tripdata_2009_small.parquet",
    2: base_dir + "yellow_tripdata_2009-01.parquet",
    3: base_dir + "yellow_tripdata_2009_large.parquet"
}

file_path = caminho[ESCALA_DATASET]

# Dicionarios globais de tempos pre-inicializados para evitar NameError no grafico final
pandas_results = {}
dask_results = {}
pyspark_results = {}
modin_results = {}
joblib_results = {}


---
## Secção 1: Profiling com Pandas

O Pandas executa em modo **eager** e single-thread. Espera-se que a maior parte do tempo de CPU seja despendida diretamente em funções internas compiladas do NumPy ou do interpretador C.

In [ ]:
df_pd = pd.read_parquet(file_path)

In [ ]:
%%prun -s tottime -l 15
# Profiling da operação de Value Counts com Pandas
import time
start = time.time()
vc_pd = df_pd['vendor_name'].value_counts()
pandas_results['Value Counts'] = time.time() - start


In [ ]:
%%prun -s tottime -l 15
# Profiling da operação de GroupBy com Pandas
import time
start = time.time()
gb_pd = df_pd.groupby('Payment_Type')['Fare_Amt'].mean()
pandas_results['GroupBy'] = time.time() - start

In [ ]:
import gc
del df_pd, vc_pd, gb_pd
gc.collect()

---
## Secção 2: Profiling com Dask

O Dask constrói um grafo de tarefas e executa-as em chunks. O profiling revelará o tempo de coordenação do agendador síncrono e a concatenação dos pedaços intermédios.

In [ ]:
# Patch de compatibilidade Dask e Python 3.11
try:
    import dask.utils
    original_derived_from = dask.utils.derived_from
    def safe_derived_from(*args, **kwargs):
        decorator = original_derived_from(*args, **kwargs)
        def safe_decorator(func):
            try:
                return decorator(func)
            except Exception:
                return func
        return safe_decorator
    dask.utils.derived_from = safe_derived_from
    import dask
    dask.config.set(scheduler='synchronous')
except Exception as e:
    print(f"Aviso Dask patch: {e}")

import dask.dataframe as dd

df_dd = dd.read_parquet(file_path)

In [ ]:
%%prun -s tottime -l 15
# Profiling da operação de Value Counts com Dask
import time
start = time.time()
vc_dd = df_dd['vendor_name'].value_counts().compute()
dask_results['Value Counts'] = time.time() - start


In [ ]:
%%prun -s tottime -l 15
# Profiling da operação de GroupBy com Dask
import time
start = time.time()
gb_dd = df_dd.groupby('Payment_Type')['Fare_Amt'].mean().compute()
dask_results['GroupBy'] = time.time() - start


In [ ]:
import gc
del df_dd, vc_dd, gb_dd
gc.collect()

---
## Secção 3: Profiling com PySpark (Koalas)

No PySpark, a computação real ocorre na JVM (Java Virtual Machine). O profiling do Python mostrará predominantemente chamadas de sockets, comunicação via rede, e serialização de dados através do protocolo Py4J.

In [ ]:
import os
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'
from pyspark.sql import SparkSession
import pyspark.pandas as ps

spark = SparkSession.builder \
    .appName('CDLE-Profiling') \
    .config('spark.sql.ansi.enabled', 'false') \
    .getOrCreate()

df_ps = ps.read_parquet(file_path)

In [ ]:
%%prun -s tottime -l 15
# Profiling da operação de Value Counts com PySpark
import time
start = time.time()
vc_ps = df_ps['vendor_name'].value_counts()
pyspark_results['Value Counts'] = time.time() - start


In [ ]:
%%prun -s tottime -l 15
# Profiling da operação de GroupBy com PySpark
import time
start = time.time()
gb_ps = df_ps.groupby('Payment_Type')['Fare_Amt'].mean()
pyspark_results['GroupBy'] = time.time() - start


In [ ]:
import gc
spark.stop()
del df_ps, vc_ps, gb_ps
gc.collect()

---
## Secção 4: Profiling com Modin

O Modin abstrai a distribuição do Pandas. Ao correr com o engine em modo python/single-process para evitar picos de memória, o profiling revelará os wrappers de metadados e distribuição criados pelo Modin.

In [ ]:
import os
os.environ["MODIN_ENGINE"] = "python"

# Patch de compatibilidade com Pandas 2.1.4
import sys
import pandas as pd
try:
    import pandas.core.arrays.arrow
except ImportError:
    import types
    sys.modules['pandas.core.arrays.arrow'] = types.ModuleType('pandas.core.arrays.arrow')
    import pandas.core.arrays.arrow

if not hasattr(pandas.core.arrays.arrow, 'ListAccessor'):
    class DummyListAccessor: pass
    pandas.core.arrays.arrow.ListAccessor = DummyListAccessor

if not hasattr(pandas.core.arrays.arrow, 'StructAccessor'):
    class DummyStructAccessor: pass
    pandas.core.arrays.arrow.StructAccessor = DummyStructAccessor

import modin.pandas as mpd

df_mod = mpd.read_parquet(file_path)

In [ ]:
%%prun -s tottime -l 15
# Profiling da operação de Value Counts com Modin
import time
start = time.time()
val_mod = df_mod['vendor_name'].value_counts()
modin_results['Value Counts'] = time.time() - start


In [ ]:
%%prun -s tottime -l 15
# Profiling da operação de GroupBy com Modin
import time
start = time.time()
gb_mod = df_mod.groupby('Payment_Type')['Fare_Amt'].mean()
modin_results['GroupBy'] = time.time() - start


In [ ]:
import gc
del df_mod, val_mod, gb_mod
gc.collect()
print("[✓] Memória Modin libertada.")

---
## Secção 5: Profiling com Joblib

O Joblib fatia o dataframe e distribui a carga por subprocessos. O profiling do processo pai mostrará o overhead de spawn/fork dos processos, serialização de dados (dumping/pickling) e coordenação da pool de processos.

In [ ]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed

print("[Joblib] A carregar dataframe base...")
df_pd_job = pd.read_parquet(file_path)
print("[OK] Dataframe carregado.")

In [ ]:
%%prun -s tottime -l 15
# Profiling da operação de Value Counts com Joblib
import time
start = time.time()
def val_chunk(chunk):
    return chunk['vendor_name'].value_counts()
chunks = np.array_split(df_pd_job, 4)
results = Parallel(n_jobs=4)(delayed(val_chunk)(chunk) for chunk in chunks)
vc_job = pd.concat(results).groupby(level=0).sum()
joblib_results['Value Counts'] = time.time() - start


In [ ]:
%%prun -s tottime -l 15
# Profiling da operação de GroupBy com Joblib
import time
start = time.time()
def gb_chunk(chunk):
    return chunk.groupby('Payment_Type')['Fare_Amt'].agg(['sum', 'count'])
chunks = np.array_split(df_pd_job, 4)
results = Parallel(n_jobs=4)(delayed(gb_chunk)(chunk) for chunk in chunks)
combined = pd.concat(results).groupby('Payment_Type').sum()
gb_job = combined['sum'] / combined['count']
joblib_results['GroupBy'] = time.time() - start


In [ ]:
import gc
del df_pd_job, vc_job, gb_job
gc.collect()

---
## Secção 6: Análise Comparativa das Operações sob Instrumentação

O bloco abaixo reúne os tempos de execução recolhidos *dentro* do ambiente de profiling. Note que estes tempos incluem o overhead introduzido pelo `cProfile`, servindo para analisar a relação de performance das operações complexas entre si.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

if 'ESCALA_DATASET' not in locals():
    ESCALA_DATASET = 1

escala_nome = {1: 'Pequena', 2: 'Média', 3: 'Grande'}.get(ESCALA_DATASET, 'Pequena')

active_profiling = {}
try:
    if 'pandas_results' in locals() and pandas_results:
        active_profiling['Pandas (s)'] = pandas_results
except NameError: pass

try:
    if 'dask_results' in locals() and dask_results:
        active_profiling['Dask (s)'] = dask_results
except NameError: pass

try:
    if 'pyspark_results' in locals() and pyspark_results:
        active_profiling['PySpark (s)'] = pyspark_results
except NameError: pass

try:
    if 'modin_results' in locals() and modin_results:
        active_profiling['Modin (s)'] = modin_results
except NameError: pass

try:
    if 'joblib_results' in locals() and joblib_results:
        active_profiling['Joblib (s)'] = joblib_results
except NameError: pass

if active_profiling:
    prof_df = pd.DataFrame(active_profiling)
    
    print('='*60)
    print(f'  TEMPOS DE EXECUÇÃO SOB PROFILING - ESCALA {escala_nome.upper()}  ')
    print('='*60)
    display(prof_df.round(3))
    
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot de barras comparativo
    prof_df.plot(kind='bar', ax=ax, width=0.7, colormap='viridis', edgecolor='black', alpha=0.9)
    
    plt.title(f'Performance de Operações sob Profiling - Escala {escala_nome} (Com Overhead do cProfile)', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Operações Analisadas', fontsize=11, fontweight='bold', labelpad=10)
    plt.ylabel('Tempo de Execução (segundos)', fontsize=11, fontweight='bold', labelpad=10)
    plt.xticks(rotation=0, fontsize=10)
    plt.yticks(fontsize=10)
    ax.yaxis.grid(True, linestyle='--', alpha=0.6)
    
    plt.legend(title='Frameworks Monitorizados', frameon=True, shadow=True, facecolor='white')
    plt.tight_layout()
    
    # Gravar o gráfico de profiling
    os.makedirs('results', exist_ok=True)
    escala_slug = {1: 'Pequena', 2: 'Media', 3: 'Grande'}.get(ESCALA_DATASET, 'Pequena')
    plot_path = f'results/CDLE_profiling_performance_{escala_slug}.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"\n[✓] Gráfico de profiling guardado com sucesso em: {plot_path}")
    
    plt.show()